# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR^2 Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described by a Croissant schema and available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and discover the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(url)

# Show dataset name and description for a quick intro
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Let's look at the record sets, their fields, and their IDs. We use the `@id` of each record set and field in code. This ensures robust referencing independent of field renaming or ordering.

In [ ]:
# List all record sets in the dataset.
dataset_record_sets = dataset.metadata.record_sets
print("Available Record Sets (by their @id):")
for rs in dataset_record_sets:
    print(f"  - {rs.id}: {rs.name}")
    # List fields
    print("    Fields (by their @id):")
    for field in rs.fields:
        print(f"      - {field.id}: {field.name} ({field.data_type})")

## 3. Data Extraction
We will extract data from one or more record sets by their `@id` into pandas DataFrames and briefly preview the data.
You may adjust the specific `@id` used based on the output of the previous overview section.

In [ ]:
# Find all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
# For this dataset, the main record set often contains the main data. Let's load all for demonstration.
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id}: {df.shape[0]} records, Columns: {df.columns.tolist()}")
# For demonstration, set the main record set id (pick first one by default)
if record_set_ids:
    main_rs_id = record_set_ids[0]
else:
    main_rs_id = None

if main_rs_id is not None:
    print(f"\nPreview of the '{main_rs_id}' DataFrame:")
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
We'll select numeric fields for analysis. All fields, columns, and groupings are referenced by their `@id` for reproducibility. Please review the lists above and replace placeholder IDs as needed based on your runs. Filtering, normalization, and grouping are standard first steps.

In [ ]:
# Pick a field to analyze (adjust field ID as appropriate)
# Use the overview above to locate numeric field (e.g., 'cr:normalized_abundance').
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    # Find numeric fields
    numeric_fields = [
        field.id for field in dataset.metadata.record_sets[0].fields
        if field.data_type in ['schema:Number', 'schema:Float', 'schema:Integer']
    ]
    print(f"Numeric fields for '{main_rs_id}': {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Proceeding with analysis using numeric field '{numeric_field}'")
        # Filter
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]}")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if available (choose non-numeric field)
        group_fields = [
            field.id for field in dataset.metadata.record_sets[0].fields
            if field.data_type not in ['schema:Number', 'schema:Float', 'schema:Integer']
        ]
        if group_fields:
            group_field = group_fields[0]
            if group_field in df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"Grouped (mean) by '{group_field}':")
                print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set to analyze.")

## 5. Visualization
Let's visualize the distribution of a key numeric field and, if relevant, compare across category groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and numeric_fields:
    df = dataframes[main_rs_id]
    numeric_field = numeric_fields[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If there is at least one categorical/grouping field
    if group_fields:
        group_field = group_fields[0]
        if group_field in df.columns:
            plt.figure(figsize=(10, 4))
            # Plot mean numeric value by group
            means = df.groupby(group_field)[numeric_field].mean().sort_values()
            sns.barplot(x=means.index, y=means.values)
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the Mass Spectrometry dataset's Croissant schema and data.
- Reviewed all record sets and their field `@id`s for robust querying.
- Extracted data into DataFrames via the `mlcroissant` API.
- Performed elementary filtering, normalization, and grouping using purely `@id`-referenced fields.
- Visualized numeric data distributions and summarized across categorical groupings.

This process demonstrates FAIR-aligned, reproducible data access and analysis. For specialist or domain-specific analysis, adjust filtering/grouping fields as appropriate.